# CASMI26 — Corrected PubChem + fixed neural 0.15

**Submission-only experiment.** Candidate-specific PubChem promotion → all existing baseline rules (including Engine 2 rank-2) → neural reranking of the final 25 with **weight 0.15, offset 3.0**. No override follows the neural stage.

## Kaggle setup
- Attach the **same input datasets and versions/wheels as your successful corrected 0.465 baseline**, plus **casmi_spectrum_model_v1.zip** (or the extracted dataset already used by your neural notebook).
- **Python 3.12 · T4 GPU · Internet OFF · fresh session.** No additional dataset or selected_policy.json is needed.
- Run **Save Version → Run All**, then submit **submission.csv**.

## Runtime
One baseline inference run and one neural scoring run. **No labeled replay, train-data hashing for validation, validation checkpoint, tuning or panel construction.** The original baseline code and its runtime/simulator budgets are preserved to avoid confounding this experiment. Public-process GPU memory is released before the neural model loads.

The inherited visible-test switch processes 12 molecules and retains placeholders for the remainder; the hidden scoring rerun processes the hidden test. Visible outputs are smoke checks, not accuracy measurements. Runtime on the hidden test depends on its size; no new LB score has been measured.

The reference file `submission_baseline0465.csv` preserves this run's corrected baseline. The experiment changes only candidate order, never the candidate set. Weight 0.15 is pre-specified for this controlled experiment, not selected by this notebook.

Credits and complete upstream code remain embedded in `fixed025_code/frozen_0465.ipynb`.

Control: the reported LB 0.466 recipe at weight 0.25. Only NEURAL_WEIGHT changes; no finite-routing change or top-1 protection.


In [ ]:
NEURAL_WEIGHT = 0.15
RRF_OFFSET = 3.0
assert NEURAL_WEIGHT == 0.15 and RRF_OFFSET == 3.0


## 1. Verify inputs and materialize the pinned source

In [ ]:
from pathlib import Path
import os,sys,json,base64,hashlib,subprocess,shutil
ROOT=Path('/kaggle/working');ROOT.mkdir(exist_ok=True);os.chdir(ROOT)
CODE=ROOT/'fixed025_code';CODE.mkdir(exist_ok=True)
assert sys.version_info[:2] == (3,12), 'Use the source notebook Python 3.12 image'
assert not (ROOT/'work/spec_meta.parquet').exists() and not (ROOT/'submission.csv').exists(), 'Start a fresh session'
MODULES = {'runner.py': '"""Execute the pinned corrected baseline once, then export its final candidates."""\nimport os,sys,json,hashlib\nfrom pathlib import Path\nfrom IPython.display import display\nif __name__==\'__main__\':\n    source=Path(sys.argv[1]); expected=sys.argv[2]; root=Path(sys.argv[3]).resolve()\n    assert hashlib.sha256(source.read_bytes()).hexdigest()==expected, \'Source hash differs\'\n    assert not os.environ.get(\'INC_PANEL\'), \'Replay input is not allowed\'\n    root.mkdir(parents=True,exist_ok=True); os.chdir(root)\n    assert not (root/\'work/spec_meta.parquet\').exists(), \'Fresh pipeline directory required\'\n    assert not (root/\'submission.csv\').exists(), \'Fresh pipeline directory required\'\n    scope={\'__name__\':\'__main__\',\'display\':display}\n    for i,c in enumerate(json.loads(source.read_text())[\'cells\']):\n        if c[\'cell_type\']!=\'code\': continue\n        print(f\'PIPELINE CELL {i} START\',flush=True)\n        exec(compile(\'\'.join(c[\'source\']),f\'pipeline_cell_{i}\',\'exec\'),scope)\n        print(f\'PIPELINE CELL {i} DONE\',flush=True)\n    assert \'candidate_promotion_evidence\' in scope, \'Candidate-specific promotion fix missing\'\n    from export_evidence import export\n    export(scope,root)\n    (root/\'execution_receipt.json\').write_text(json.dumps(dict(source_sha256=expected,\n        mode=\'submission\',source_adaptations=[],candidate_promotion_fix=True,\n        is_full_run=bool(scope[\'IS_RERUN\']),processed_molecules=int(scope[\'te\'].molecule_id.nunique())),indent=2))\n', 'export_evidence.py': '"""Executed in the frozen pipeline process AFTER every original final rule."""\nimport math,json,shutil\nfrom pathlib import Path\nimport pandas as pd\n\ndef export(scope, root):\n    root=Path(root);frame=pd.read_csv(root/\'submission.csv\',dtype=str,keep_default_na=False)\n    shutil.copy2(root/\'submission.csv\',root/\'submission_baseline0465.csv\')\n    processed=set(scope[\'te\'].molecule_id.astype(str));identity=scope[\'metric_identity\']\n    def rank_map(smis):\n        out={}\n        for rank,s in enumerate(smis,1):\n            key=identity(s)\n            if key is not None:out.setdefault(key,rank)\n        return out\n    def finite(v):\n        try:return float(v) if v is not None and math.isfinite(float(v)) else None\n        except (ValueError,TypeError):return None\n    rows=[]\n    for mid,answer in frame.itertuples(index=False):\n        candidates=answer.split(\';\')\n        assert 1<=len(candidates)<=25 and len(candidates)==len(set(candidates)),mid\n        active=mid in processed\n        base=scope[\'BASE\'].get(mid,([],[],0.))\n        maps={\n            \'main\':rank_map(base[0]),\n            \'prefusion\':rank_map(scope.get(\'PREFUSION_LISTS\',{}).get(mid,[])),\n            \'engine2\':rank_map(scope.get(\'ENG\',{}).get(mid,{}).get(\'smiles\',[])),\n            \'pubchem\':rank_map(scope.get(\'PC\',{}).get(mid,{}).get(\'pc\',[]))}\n        ice=scope.get(\'ICE_SCORES\',{}).get(mid,{}) or {}\n        gl=scope[\'gl_of\'](mid) or {}\n        for rank,s in enumerate(candidates,1):\n            ident=identity(s);key=scope[\'chem\'].score_key(s)\n            row=dict(molecule_id=mid,smiles=s,metric_key=ident[0] if ident else None,baseline_rank=rank,\n                processed=active,library_max=finite(base[2]) or 0.,proxy_gl=mid in scope.get(\'N1_GL_MIDS\',set()),\n                ice=finite(ice.get(s,ice.get(key))),gl=finite(gl.get(s,gl.get(key))))\n            row.update({name+\'_rank\':mapping.get(ident,0) for name,mapping in maps.items()});rows.append(row)\n    pd.DataFrame(rows).to_parquet(root/\'frozen_candidate_evidence.parquet\',index=False)\n    # Save the effective query spectra used by the pipeline (12 rows\' molecules in visible smoke, all in rerun).\n    scope[\'te\'].to_parquet(root/\'effective_query_spectra.parquet\',index=False)\n    (root/\'frozen_run_info.json\').write_text(json.dumps(dict(processed_molecules=len(processed),\n        output_molecules=len(frame),is_full_run=bool(scope[\'IS_RERUN\']),\n        note=\'Baseline file saved after the original Engine2 rank-2 insertion; candidate-specific PubChem promotion fix included.\'),indent=2))\n    print(\'EXPORTED FROZEN FINAL25\',len(rows),\'candidate rows\',flush=True)\n', 'neural_scores.py': '"""Runs after the public pipeline process exits, freeing its GPU memory."""\nimport json,sys,importlib.util,types,hashlib\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch\n\ndef load_model(bundle,checkpoint,device):\n    # Private module namespace prevents collision with public engines\' model/features modules.\n    package=\'_casmi_own\';p=types.ModuleType(package);p.__path__=[str(bundle)];sys.modules[package]=p\n    for name in [\'features\',\'model\']:\n        spec=importlib.util.spec_from_file_location(package+\'.\'+name,Path(bundle)/(name+\'.py\'))\n        mod=importlib.util.module_from_spec(spec);sys.modules[spec.name]=mod;spec.loader.exec_module(mod)\n    features=sys.modules[package+\'.features\'];cls=sys.modules[package+\'.model\'].SpectrumCandidateModel\n    cfg=json.loads((Path(bundle)/\'config.json\').read_text());model=cls(cfg).to(device)\n    ck=torch.load(Path(bundle)/checkpoint,map_location=device,weights_only=True)\n    model.load_state_dict(ck[\'state_dict\']);model.eval()\n    return model,features,cfg\n\n@torch.inference_mode()\ndef score(bundle,checkpoint,evidence,queries,output,device=\'cuda\'):\n    torch.set_num_threads(4)\n    if device==\'cuda\':assert torch.cuda.is_available(),\'GPU required for neural scoring\'\n    model,feat,cfg=load_model(bundle,checkpoint,device)\n    frame=pd.read_parquet(evidence);q=pd.read_parquet(queries);q[\'molecule_id\']=q.molecule_id.astype(str)\n    query={mid:g for mid,g in q.groupby(\'molecule_id\',sort=False)};cache={};parts=[];errors=[]\n    for index,(mid,g) in enumerate(frame.groupby(\'molecule_id\',sort=False)):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\').copy()\n        g[\'neural_ok\']=False\n        if not g.processed.iloc[0]:\n            for c in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[c]=0.\n            parts.append(g);continue\n        try:\n            assert mid in query,mid\n            records=[]\n            # Use EVERY observed query spectrum; no adduct merging and no new spectrum selection rule.\n            for row in query[mid].itertuples(index=False):\n                try:\n                    pk,n=feat.peaks(row.ms2_mzs,row.ms2_normalized_intensities,row.precursor_mz,cfg[\'max_peaks\'])\n                    if n<cfg[\'min_peaks\']:continue\n                    m,a=feat.metadata(row.precursor_mz,row.adduct,row.ionization_mode,row.collision_energy_ev)\n                    records.append((pk,n,m,a))\n                except (ValueError,TypeError):continue\n            if not records:raise ValueError(\'No usable spectra after the training preprocessing filters\')\n            fp=[]\n            for s in g.smiles:\n                if s not in cache:cache[s]=feat.structure(s,cfg[\'fp_bits\'])[-1]\n                fp.append(cache[s])\n            F=torch.tensor(np.stack(fp),device=device,dtype=torch.float32);M=model.encode_molecules(F)\n            cos_all=[];fp_all=[]\n            for st in range(0,len(records),64):\n                b=records[st:st+64]\n                x=torch.tensor(np.stack([v[0] for v in b]),device=device)\n                lengths=torch.tensor([v[1] for v in b],device=device,dtype=torch.long)\n                meta=torch.tensor(np.stack([v[2] for v in b]),device=device)\n                adduct=torch.tensor([v[3] for v in b],device=device,dtype=torch.long)\n                Z,P=model.encode_spectra(x,lengths,meta,adduct)\n                cos_all.append((Z@M.T).cpu().numpy());fp_all.append((P@F.T/cfg[\'fp_bits\']).cpu().numpy())\n            cos=np.concatenate(cos_all);fps=np.concatenate(fp_all)\n            combined=cos/cfg[\'temperature\']+cfg[\'fp_score_weight\']*fps\n            assert np.isfinite(combined).all(),mid\n            g[\'neural_cos\']=cos.mean(0);g[\'neural_fp\']=fps.mean(0);g[\'neural_score\']=combined.mean(0)\n            g[\'neural_std\']=combined.std(0);g[\'n_spectra\']=len(records);g[\'n_adducts\']=query[mid].adduct.nunique()\n            g[\'neural_ok\']=True\n            parts.append(g)\n        except (ValueError,TypeError) as exc:\n            errors.append(dict(molecule_id=mid,error=str(exc)))\n            for col in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[col]=0.\n            parts.append(g)\n        if (index+1)%25==0:print(\'NEURAL SCORED\',index+1,\'molecules\',flush=True)\n    result=pd.concat(parts,ignore_index=True)\n    assert len(result)==len(frame)\n    result.to_parquet(output,index=False)\n    Path(str(output)+\'.errors.json\').write_text(json.dumps(errors,indent=2))\n    print(\'NEURAL input failures kept at baseline:\',len(errors),flush=True)\n    print(\'NEURAL FEATURES\',result.shape,checkpoint,flush=True)\n\nif __name__==\'__main__\':score(*sys.argv[1:])\n', 'submission.py': 'import json,hashlib\nfrom pathlib import Path\nimport pandas as pd,numpy as np\nfrom ranking import rrf_order\n\ndef apply(evidence,baseline,output,weight=.25,offset=3.):\n    f=pd.read_parquet(evidence);base=pd.read_csv(baseline,dtype=str,keep_default_na=False)\n    groups={mid:g for mid,g in f.groupby(\'molecule_id\',sort=False)};rows=[];audit=[]\n    assert base.molecule_id.is_unique and set(groups)==set(base.molecule_id)\n    for mid,text in base.itertuples(index=False):\n        g=groups[mid].sort_values(\'baseline_rank\',kind=\'stable\');original=text.split(\';\')\n        assert g.smiles.tolist()==original,mid\n        active=bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n        order=rrf_order(g.neural_score.values,weight,offset) if active else np.arange(len(g))\n        new=g.smiles.iloc[order].tolist()\n        assert len(new)==len(original) and set(new)==set(original),mid\n        assert len(new)<=25 and len(set(new))==len(new)\n        rows.append((mid,\';\'.join(new)))\n        audit.append(dict(molecule_id=mid,processed=active,changed=new!=original,\n            changed_top1=new[0]!=original[0],candidate_count=len(new),weight=weight))\n    result=pd.DataFrame(rows,columns=[\'molecule_id\',\'smiles\'])\n    assert result.molecule_id.tolist()==base.molecule_id.tolist()\n    result.to_csv(output,index=False)\n    out=Path(output).parent;pd.DataFrame(audit).to_csv(out/\'neural_rerank_audit.csv\',index=False)\n    report=dict(method=\'baseline rank + neural rank reciprocal-rank fusion AFTER every baseline rule\',\n        neural_weight=weight,rrf_offset=offset,weight_is_validated=False, validation_scope="Fixed-weight submission experiment; no validation performed",\n        candidate_set_preserved=True,changed_rows=sum(x[\'changed\'] for x in audit),\n        changed_top1=sum(x[\'changed_top1\'] for x in audit),rows=len(rows),\n        frozen_sha256=hashlib.sha256(Path(baseline).read_bytes()).hexdigest(),\n        experimental_sha256=hashlib.sha256(Path(output).read_bytes()).hexdigest())\n    (out/\'neural_submission_report.json\').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n    return report\n', 'ranking.py': '"""Pure deterministic ranking utilities shared by both notebooks."""\nimport numpy as np\nimport pandas as pd\n\ndef stable_rank(values):\n    values=np.asarray(values,dtype=float)\n    assert np.isfinite(values).all(), \'Nonfinite candidate scores\'\n    order=np.argsort(-values,kind=\'stable\');ranks=np.empty(len(values),int);ranks[order]=np.arange(1,len(values)+1)\n    return ranks\n\ndef rrf_order(neural_scores,weight=0.15,offset=3.0):\n    assert weight>=0 and offset>=0\n    n=len(neural_scores);baseline=np.arange(1,n+1)\n    # Zero weight guarantees exact baseline order even when neural scores are unavailable.\n    if weight==0:return np.arange(n)\n    ranks=stable_rank(neural_scores)\n    return np.argsort(-(1/(offset+baseline)+weight/(offset+ranks)),kind=\'stable\')\n\ndef metrics(frame,score=\'prediction\'):\n    out=[]\n    for mid,g in frame.groupby(\'molecule_id\',sort=False):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\')\n        order=np.argsort(-g[score].to_numpy(),kind=\'stable\')\n        labels=g.label.to_numpy()[order];where=np.flatnonzero(labels)\n        r=int(where[0]+1) if len(where) else None\n        out.append(dict(molecule_id=mid,mrr25=1/r if r and r<=25 else 0.,top1=int(r==1),\n                        candidate_recall=int(bool(len(where))),rank=r))\n    return pd.DataFrame(out)\n\nBASE_FEATURES=[\'baseline_rr\',\'main_rr\',\'prefusion_rr\',\'engine2_rr\',\'pubchem_rr\',\'engine_agreement\',\n    \'library_max\',\'proxy_gl\',\'ice_missing\',\'gl_missing\',\'ice_z\',\'gl_z\',\n    \'neural_cos_z\',\'neural_fp_z\',\'neural_score_z\',\'neural_rr\',\'neural_std\',\n    \'n_spectra_log\',\'n_adducts\',\'neural_top_gap\',\'neural_agrees_top1\',\'engine2_agrees_top1\',\n    \'neural_delta_rank\',\'neural_x_library\',\'neural_x_agreement\',\'neural_x_sim_missing\',\n    \'main_missing\',\'engine2_missing\',\'pubchem_missing\']\n\ndef make_features(frame):\n    f=frame.copy()\n    for src in [\'baseline\',\'main\',\'prefusion\',\'engine2\',\'pubchem\']:\n        r=pd.to_numeric(f[src+\'_rank\'],errors=\'coerce\');f[src+\'_rr\']=np.where(r>0,1/(3+r),0)\n        if src in [\'main\',\'engine2\',\'pubchem\']:f[src+\'_missing\']=(~(r>0)).astype(float)\n    for src in [\'ice\',\'gl\',\'neural_cos\',\'neural_fp\',\'neural_score\']:\n        v=pd.to_numeric(f[src],errors=\'coerce\');v=v.where(np.isfinite(v))\n        if src in [\'ice\',\'gl\']:f[src+\'_missing\']=v.isna().astype(float)\n        grp=v.groupby(f.molecule_id,sort=False)\n        f[src+\'_z\']=((v-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    f[\'neural_rank\']=0.;f[\'neural_top_gap\']=0.;f[\'neural_agrees_top1\']=0.;f[\'engine2_agrees_top1\']=0.\n    for _,g in f.groupby(\'molecule_id\',sort=False):\n        idx=g.index;rank=stable_rank(g.neural_score.values);f.loc[idx,\'neural_rank\']=rank\n        vals=np.sort(g.neural_score_z.values)[::-1]\n        f.loc[idx,\'neural_top_gap\']=float(vals[0]-vals[1]) if len(vals)>1 else 0.\n        f.loc[idx,\'neural_agrees_top1\']=float(g.iloc[np.argmin(rank)].baseline_rank==1)\n        f.loc[idx,\'engine2_agrees_top1\']=float(((g.baseline_rank==1)&(g.engine2_rank==1)).any())\n    f[\'neural_rr\']=1/(3+f.neural_rank)\n    f[\'engine_agreement\']=((f.main_rank>0).astype(int)+(f.engine2_rank>0).astype(int)+(f.pubchem_rank>0).astype(int))\n    f[\'n_spectra_log\']=np.log1p(f.n_spectra)\n    f[\'neural_delta_rank\']=(f.baseline_rank-f.neural_rank)/25\n    f[\'neural_x_library\']=f.neural_score_z*f.library_max\n    f[\'neural_x_agreement\']=f.neural_score_z*f.engine_agreement\n    f[\'neural_x_sim_missing\']=f.neural_score_z*f.ice_missing*f.gl_missing\n    X=f[BASE_FEATURES].astype(float).replace([np.inf,-np.inf],np.nan).fillna(0.)\n    assert np.isfinite(X.values).all()\n    return X\n\ndef residual_scores(frame,probability,beta):\n    p=np.clip(np.asarray(probability,float),1e-5,1-1e-5)\n    raw=pd.Series(np.log(p/(1-p)),index=frame.index)\n    grp=raw.groupby(frame.molecule_id,sort=False)\n    z=((raw-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    correction=float(beta)*z.values\n    if \'neural_ok\' in frame:correction=np.where(frame.neural_ok.values,correction,0.)\n    return -np.log(frame.baseline_rank.values.astype(float))+correction\n', 'bundle.py': "import hashlib,json,zipfile\nfrom pathlib import Path\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,'rb') as f:\n        for b in iter(lambda:f.read(8<<20),b''):h.update(b)\n    return h.hexdigest()\n\ndef find_bundle(input_root,output_root,pins):\n    roots=[]\n    for path in Path(input_root).rglob('final_model.pt'):\n        if (path.parent/'config.json').is_file():roots.append(path.parent)\n    if not roots:\n        archives=list(Path(input_root).rglob('casmi_spectrum_model_v1.zip'))\n        assert len(archives)==1, 'Attach the supplied casmi_spectrum_model_v1.zip as a Kaggle dataset (zipped or extracted)'\n        dest=Path(output_root)/'uploaded_model';dest.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(archives[0]) as z:\n            for item in z.infolist():\n                target=(dest/item.filename).resolve()\n                assert target.is_relative_to(dest.resolve()),'Unsafe ZIP member'\n            z.extractall(dest)\n        roots=[p.parent for p in dest.rglob('final_model.pt')]\n    matching=[r for r in roots if all((r/n).is_file() and sha(r/n)==h for n,h in pins.items())]\n    assert len(matching)==1, f'Expected exactly one matching model bundle, found {len(matching)}. Use the attached v1 artifacts.'\n    return matching[0]\n"}
MODEL_PINS = {'features.py': 'a97ad9b3ab3711517864d56e497e1672cb29bb05683c72b52db9f9b0a17b1455', 'model.py': '2547418857386d1c2513fd1ab6c70961e6a96e73d9d495bbfe3052197e69dc39', 'config.json': '563fb2a2a81ca48ea094f9ab78c157a48a6e9a5503e85e6ce8f0fb3615c1cc03', 'final_model.pt': '44436b2daec64373e66f9120dd2389c24fc83dd9430ebeeac18e3cb49dcb9642'}
FROZEN_SHA256='aa1a43cd050f5428cad36a82143555422bd897d1ae48ebfc733f90ce67b88743'
FROZEN_B64='...'
for name,text in MODULES.items():(CODE/name).write_text(text)
FROZEN=CODE/'frozen_0465.ipynb';FROZEN.write_bytes(base64.b64decode(FROZEN_B64))
assert hashlib.sha256(FROZEN.read_bytes()).hexdigest()==FROZEN_SHA256
sys.path.insert(0,str(CODE))
from bundle import find_bundle,sha
BUNDLE=find_bundle('/kaggle/input',ROOT,MODEL_PINS)
print('Verified bundle:',BUNDLE,'\nCorrected source:',FROZEN_SHA256)
# Resolve the official input explicitly, not an arbitrary mounted train.parquet.
COMP=next((p for p in [Path('/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'),
    Path('/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra')] if (p/'train.parquet').is_file()),None)
assert COMP is not None, 'Competition input is missing'
def run_logged(args,log,extra_env=None):
    env={**os.environ,'PYTHONHASHSEED':'0','PYTHONUNBUFFERED':'1'};env.pop('INC_PANEL',None)
    env.update(extra_env or {})
    with open(log,'w') as handle:
        p=subprocess.Popen([str(x) for x in args],cwd=ROOT,env=env,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            handle.write(line);handle.flush()
            # Retain full logs, suppress only giant SVG drawing dumps in notebook output.
            if len(line)<5000:print(line,end='',flush=True)
        status=p.wait()
    if status:raise RuntimeError(f'Process failed ({status}); inspect {log}')
# Install exactly the original notebook's offline RDKit wheel before any chemistry imports.
bootstrap=''.join(json.loads(FROZEN.read_text())['cells'][2]['source'])
(CODE/'bootstrap.py').write_text(bootstrap)
run_logged([sys.executable,CODE/'bootstrap.py'],ROOT/'bootstrap.log')

## 2. Run the corrected baseline once

In [ ]:
run_logged([sys.executable,CODE/'runner.py',FROZEN,FROZEN_SHA256,ROOT],ROOT/'submission_pipeline.log')

## 3. Score only the baseline final candidates

In [ ]:
run_logged([sys.executable,CODE/'neural_scores.py',BUNDLE,'final_model.pt',
    ROOT/'frozen_candidate_evidence.parquet',ROOT/'effective_query_spectra.parquet',
    ROOT/'neural_candidate_features.parquet','cuda'],ROOT/'submission_neural.log')

## 4. Apply fixed 0.15 fusion and verify the submission

In [ ]:
import pandas as pd
from IPython.display import display,FileLink
from submission import apply
assert NEURAL_WEIGHT == 0.15 and RRF_OFFSET == 3.0
REPORT=apply(ROOT/'neural_candidate_features.parquet',ROOT/'submission_baseline0465.csv',
    ROOT/'submission.csv',weight=NEURAL_WEIGHT,offset=RRF_OFFSET)
REPORT.update(experiment='corrected_pubchem_fixed_neural_015',
    policy_selection='fixed; no validation or tuning',checkpoint='final_model.pt',
    corrected_baseline_source_sha256=FROZEN_SHA256)
(ROOT/'neural_submission_report.json').write_text(json.dumps(REPORT,indent=2))
result=pd.read_csv(ROOT/'submission.csv',dtype=str,keep_default_na=False)
reference=pd.read_csv(ROOT/'submission_baseline0465.csv',dtype=str,keep_default_na=False)
expected=pd.read_csv(COMP/'sample_submission.csv',dtype=str)
assert list(result.columns)==['molecule_id','smiles']
assert result.molecule_id.is_unique
assert result.molecule_id.tolist()==expected.molecule_id.tolist()==reference.molecule_id.tolist()
assert result.smiles.str.split(';').map(len).between(1,25).all()
for before,after in zip(reference.smiles,result.smiles):
    assert sorted(before.split(';'))==sorted(after.split(';')), 'Candidate set changed'
receipt=json.loads((ROOT/'execution_receipt.json').read_text())
print('Full hidden run:',receipt['is_full_run'],'| Processed molecules:',receipt['processed_molecules'])
print('Fixed neural weight:',NEURAL_WEIGHT,'| Submit: /kaggle/working/submission.csv')
display(result.head())
for name in ['submission.csv','submission_baseline0465.csv','neural_submission_report.json',
             'neural_rerank_audit.csv','pubchem_promotion_audit.csv','execution_receipt.json']:
    display(FileLink(str(ROOT/name)))
